In [124]:
from urllib import response

#LOAD ENV VARIABLES
from dotenv import load_dotenv

load_dotenv()

#Create an API client
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
model = "gemini-3.1-flash-lite"
#noinspection PyTypeChecker
def add_user_message(messages,content):
    if isinstance(content, str):
        # Handle standard text input from the user
        messages.append(
            types.Content(
                role="user",
                parts=[types.Part.from_text(text=content)]
            )
        )
    elif isinstance(content, list):
        # Handle the list of tool result Parts returned by run_tools()
        messages.append(
            types.Content(
                role="user",
                parts=content
            )
        )

def add_assistant_message(messages,output):
    # Extracts the pre-formatted Content object from Gemini's first candidate
    if output.candidates and output.candidates[0].content:
        messages.append(output.candidates[0].content)

def chat_stream(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):

    if stop_sequences is None:
        stop_sequences = []
    params = {
        "model":model,
        "contents":messages
    }

    config = types.GenerateContentConfig(temperature=temperature)

    if system:
        config.system_instruction = system

    if stop_sequences:
        config.stop_sequences = stop_sequences

    if tools:
        config.tools = [types.Tool(function_declarations=tools)]
        # Forcing execution mode using the correct SDK configuration layout
        config.tool_config = types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="AUTO"
            )
        )

    params["config"] = config

    message = client.models.generate_content(**params)
    return message

def text_from_message(message):
    # Returns the combined text, or an empty string if no text exists
    return message.text if message.text else ""

In [125]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def  set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "datetime_str": {
                "type": "STRING",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "NUMBER",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "STRING",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "STRING",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "content": {
                "type": "STRING",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "STRING",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "invocations": {
                "type": "ARRAY",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "OBJECT",
                    "properties": {
                        "name": {
                            "type": "STRING",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "STRING",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [126]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = types.FunctionDeclaration(
    name="get_current_datetime",
    description="RReturns the current date and time formatted according to the specified format string. This tool provides the current system time formatted as a string. Use this tool when you need to know the current date and time, such as for timestamping records, calculating time differences, or displaying the current time to users. The default format returns the date and time in ISO-like format (YYYY-MM-DD HH:MM:SS).",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "date_format":types.Schema(
                type=types.Type.STRING,
                description="A string specifying the format of the returned datetime. Uses Python's strftime format codes. For example, '%Y-%m-%d' returns just the date in YYYY-MM-DD format, '%H:%M:%S' returns just the time in HH:MM:SS format, '%B %d, %Y' returns a date like 'May 07, 2025'. The default is '%Y-%m-%d %H:%M:%S' which returns a complete timestamp like '2025-05-07 14:32:15'.",
                default="%Y-%m-%d %H:%M:%S"
                )
        },
        required=[]
    )
)

In [127]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    return None


def run_tools(message):
    tool_result_parts = []

    # 1. Gemini stores requested tools directly in response.function_calls
    if not message.function_calls:
        return tool_result_parts

    for tool_request in message.function_calls:
        try:
            # tool_request.name is the function name string
            # tool_request.args is a Python dict containing the arguments
            tool_output = run_tool(tool_request.name, tool_request.args)

            # Successful response payload structure
            response_payload = {"result": tool_output}

        except Exception as e:
            # Error response payload structure
            response_payload = {"error": str(e)}

        # 2. Package the result into Gemini's official Part format
        tool_result_part = types.Part.from_function_response(
            name=tool_request.name,
            response=response_payload
        )

        tool_result_parts.append(tool_result_part)

    return tool_result_parts

In [128]:
def run_conversation(messages):
    while True:
        output = chat(
            messages=messages,
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema,
            ]
        )

        # 1. Save Gemini's response structure to your chat history
        add_assistant_message(messages, output)

        # 2. Print any text Gemini sent alongside the tool call
        if output.text:
            print(text_from_message(output))

        # 3. GEMINI STOP CONDITION:
        # If Gemini didn't ask to run a tool, the conversation step is done!
        if not output.function_calls:
            break

        tool_results = run_tools(output)
        add_user_message(messages, tool_results)

    return messages

In [130]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.",
)
run_conversation(messages)

----
Setting the following reminder for 2026-12-01T09:00:00:
Doctor's appointment
----
OK. I've set a reminder for your doctor's appointment on December 1, 2026, at 9:00 AM.


[Content(
   parts=[
     Part(
       text='Set a reminder for my doctors appointment after 177 days'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'date_format': '%Y-%m-%d'
         },
         id='ieAttb73',
         name='get_current_datetime'
       ),
       thought_signature=b'\x124\n2\x01\x0c9\xd6\xc7\x04\xcda\x96\x95\xf5Y\xfe\xed\xd7\x8b\x81\xd5\xe1\x01\xcf\xa6\xd4Q\x0cx\xf58)\xc3%\xf3\xf2\x0b\xf3\xe8H0\xd0h\xbeJo\xa3|\xc5\x03Mk\xc7'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='get_current_datetime',
         response={
           'result': '2026-06-07'
         }
       )
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'datetime_str': '2026-06-07',
           'duration': 177,
           'input_format': '%Y-%m-%d',
           'unit': 'days'
 